[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/advanced/08_Reasoning_and_Inference.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 推理与推断（Reasoning and Inference）

## 概览

用 **Semantica 推理模块**构建知识图谱、定义规则、执行前向/反向链、并为 AI 推理生成解释。

**文档**：[API 参考](https://semantica.readthedocs.io/reference/reasoning/)

### 先看主线

23 篇讲过推理的三段——**推得出 → 证得了 → 讲得清**。本篇同一套，两处升级：一是 `Reasoner` 直接吃 `GraphBuilder` 的图谱产物（建图与推理无缝衔接）；二是后半部的 **Deep Dive** 细讲规则语法、事实格式与双链策略。前半五步快速走读，后半沉下去练。

**你将练到**
- 定义带变量与谓词的规则
- 以谓词形式装载事实
- 跑前向与反向链
- 生成人类可读的解释

## 安装

从 PyPI 安装 Semantica：

```bash
pip install semantica
# 或安装全部可选依赖：
pip install semantica[all]
```

## 工作流：建图 → 定义规则 → 前向/反向链 → 生成解释

In [ ]:
!pip install -qU semantica


In [ ]:
from semantica.kg import GraphBuilder
from semantica.reasoning import Reasoner, ExplanationGenerator


## 第 1 步：构建知识图谱

In [ ]:
builder = GraphBuilder()

entities = [
    {"id": "alice", "type": "Person", "name": "Alice"},
    {"id": "bob", "type": "Person", "name": "Bob"},
    {"id": "charlie", "type": "Person", "name": "Charlie"},
    {"id": "sf", "type": "Location", "name": "San Francisco"},
    {"id": "california", "type": "Location", "name": "California"},
]

relationships = [
    {"source": "alice", "target": "bob", "type": "parent_of"},
    {"source": "bob", "target": "charlie", "type": "parent_of"},
    {"source": "sf", "target": "california", "type": "located_in"},
    {"source": "alice", "target": "sf", "type": "lives_in"},
]

knowledge_graph = builder.build([{"entities": entities, "relationships": relationships}])


## 第 2 步：定义规则

用逻辑语法定义推理规则——注意 `?` 前缀的变量在规则内跨谓词统一。

In [ ]:
# Initialize Reasoner
reasoner = Reasoner()

# Define rules using logic syntax
rules = [
    "IF parent_of(?a, ?b) AND parent_of(?b, ?c) THEN grandparent_of(?a, ?c)",
    "IF lives_in(?x, ?y) AND located_in(?y, ?z) THEN lives_in(?x, ?z)"
]

for rule in rules:
    reasoner.add_rule(rule)


## 第 3 步：前向链

执行前向链推导新事实。`Reasoner` 可以直接从知识图谱或事实列表推断——建图与推理无缝衔接。

In [ ]:
# Perform forward chaining to derive new facts
# The Reasoner can infer facts directly from the knowledge graph or a list of facts
inferred_facts = reasoner.infer_facts(knowledge_graph)

print(f"Inferred {len(inferred_facts)} new facts:")
for fact in inferred_facts:
    print(f" - {fact}")


## 第 4 步：反向链

定义一个待证目标，反向链倒着找证明。

In [ ]:
# Define a goal to prove
goal = "grandparent_of(alice, charlie)"

# Perform backward chaining
proof = reasoner.backward_chain(goal)

if proof:
    print(f"Goal '{goal}' proven successfully!")
else:
    print(f"Could not prove goal '{goal}'.")


## 第 5 步：生成解释

推理的"讲得清"环节：把证明变成自然语言——22 篇的溯源纪律延伸到推理层。

In [ ]:
generator = ExplanationGenerator()

# If we have a proof from backward chaining, explain it
if proof:
    proof_explanation = generator.generate_explanation(proof)
    print("Explanation for backward chaining proof:")
    print(proof_explanation.natural_language)


## 小结

推理与推断工作流：
- 知识图谱已构建
- 推理规则已定义
- 事实已装入引擎
- 前向链已执行
- 反向链已执行
- 解释已生成

---

### 深入：推理模块

这一节深入讲 Semantica 的推理能力：规则语法、事实格式、链式策略与解释生成，配健壮、可复现的示例。

**你将练习**
- 定义带变量和谓词的规则
- 以谓词形式装载事实
- 跑前向和反向链
- 生成人类可读的解释

In [ ]:
from semantica.kg import GraphBuilder
from semantica.reasoning import Reasoner, ExplanationGenerator

builder = GraphBuilder()
reasoner = Reasoner()
explainer = ExplanationGenerator()


### 规则语法

规则用谓词逻辑，变量以 `?` 为前缀。

- 例：`IF parent_of(?a, ?b) AND parent_of(?b, ?c) THEN grandparent_of(?a, ?c)`
- 变量在同一条规则内跨谓词统一（合一）
- 条件匹配时，结论作为新事实加入

In [ ]:
entities = [
    {"id": "alice", "type": "Person", "name": "Alice"},
    {"id": "bob", "type": "Person", "name": "Bob"},
    {"id": "charlie", "type": "Person", "name": "Charlie"},
    {"id": "sf", "type": "Location", "name": "San Francisco"},
    {"id": "california", "type": "Location", "name": "California"}
]

relationships = [
    {"source": "alice", "target": "bob", "type": "parent_of"},
    {"source": "bob", "target": "charlie", "type": "parent_of"},
    {"source": "sf", "target": "california", "type": "located_in"},
    {"source": "alice", "target": "sf", "type": "lives_in"}
]

knowledge_graph = builder.build([{"entities": entities, "relationships": relationships}])
print(len(knowledge_graph.get("entities", [])))
print(len(knowledge_graph.get("relationships", [])))


In [ ]:
rules = [
    "IF parent_of(?a, ?b) AND parent_of(?b, ?c) THEN grandparent_of(?a, ?c)",
    "IF lives_in(?x, ?y) AND located_in(?y, ?z) THEN lives_in(?x, ?z)"
]
for r in rules:
    reasoner.add_rule(r)


In [ ]:
for rel in relationships:
    fact = f"{rel['type']}({rel['source']}, {rel['target']})"
    reasoner.add_fact(fact)

derived = reasoner.forward_chain()
print(len(derived))
for d in derived:
    print(d.conclusion)


In [ ]:
goals = [
    "grandparent_of(alice, charlie)",
    "lives_in(alice, california)"
]
for g in goals:
    proof = reasoner.backward_chain(g)
    print(g)
    print(bool(proof))


In [ ]:
if derived:
    exp = explainer.generate_explanation(derived[0])
    print(exp.natural_language)

goal = "grandparent_of(alice, charlie)"
proof = reasoner.backward_chain(goal)
if proof:
    pexp = explainer.generate_explanation(proof)
    print(pexp.natural_language)
